# 1. Data cleaning (from raw Crunchbase/CORDIS/registry data to the analysis table)

Executed reviewer walkthrough of the data build: **every load, clean, match and
merge step shown individually with before/after counts, the reason for each
exclusion, and descriptive statistics**, as if done by hand. The scripts in
`scripts/` + `src/` remain the source of truth: this notebook calls the SAME
functions step by step and finishes by asserting cell-for-cell equality with
`data/processed/firms_analysis.parquet`.

Flow, per source: load Crunchbase, explore, clean; load registry and crawl
data, match; load grant data, clean and exclude, match; then merge, apply the
row policy, validate and catalogue the variables.

*Circular Startup Financing. Master's thesis, 2026.*

In [1]:
import os, pathlib, subprocess, sys
if pathlib.Path.cwd().name == "analysis":
    os.chdir(pathlib.Path.cwd().parent)          # notebooks execute from analysis/
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "scripts"))
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

def run(cmd):                                     # only for the end-gates (template style)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("\n".join(r.stdout.splitlines()[-12:]))
    if r.returncode != 0:
        print(r.stderr[-1500:]); raise SystemExit(r.returncode)

def overview(df, name, top=8):
    """Compact per-step descriptives: shape, dtypes, worst missingness."""
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    miss = df.isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0].head(top)
    if len(miss):
        print("  hoechste Missing-Anteile:")
        for k, v in miss.items():
            print(f"    {k:<38} {v:6.1%}")
    else:
        print("  keine fehlenden Werte")

## 1.A The raw Crunchbase data: what do we actually have?

Four files in `data/raw/` (read-only, never modified):

| File | Level | Content |
|---|---|---|
| `companies_lean.csv` / `companies_fat.csv` | **1 row per firm** | end-state snapshot (master data, descriptions, classifier columns; *fat* = all columns incl. text, *lean* = core columns) |
| `panel_lean.csv` / `panel_fat.csv` | **1 row per firm×month** | monthly events (funding events, amounts, types; *fat* = 31 GB with all columns) |

So: **companies and panel contain the same firms on different levels**, a
firm-level snapshot vs a dated monthly history. Outcomes may ONLY come from
the dated panel (end-state columns would be leakage, see 1.B).
Key everywhere: `crunchbase_url`.

In [2]:
# companies_lean: small enough to load fully. Check structure and key.
lean = pd.read_csv("data/raw/companies_lean.csv", dtype=str)
overview(lean, "companies_lean")
lean.head(3).iloc[:, :8]

companies_lean: 25,421 rows x 34 cols
  hoechste Missing-Anteile:
    cbcsv_number_of_acquisitions            97.1%
    cbm_confidence                          82.1%
    cbm_reasoning                           82.1%
    cbcsv_patents_granted                   65.8%
    cbcsv_estimated_revenue_range           44.6%
    cbcsv_number_of_articles                37.3%
    cbcsv_postal_code                       36.4%
    cbcsv_number_of_founders                25.1%


,crunchbase_url,cb_company_name,cb_website,cb_founded_year,cb_categories,cb_operating_status,cb_source_file,cb_description
0,https://www.crunchbase.com/organization/0-mission,0-Mission,https://www.the0mission.dk,2022,"Energy, Marketplace, Renewable Energy, Small a...",Active,DK.csv,0-Mission is a B2B marketplace that connects d...
1,https://www.crunchbase.com/organization/011h,011h,https://www.011h.com/,2020,"Architecture, Construction",Active,ES2.csv,011h is a construction company that specialize...
2,https://www.crunchbase.com/organization/01green,01Green,NaN,2020,Manufacturing,Active,IT1.csv,01Green is an innovative business focusing on ...


In [3]:
# companies_fat: 382 MB. Column list plus 5 example rows; row count via the key column.
fat_cols = pd.read_csv("data/raw/companies_fat.csv", nrows=0).columns
fat_ids = pd.read_csv("data/raw/companies_fat.csv", usecols=["crunchbase_url"], dtype=str)
print(f"companies_fat: {len(fat_ids):,} rows x {len(fat_cols)} cols")
print("nur in fat (nicht in lean):", sorted(set(fat_cols) - set(lean.columns))[:12], "...")
print("Firmen identisch mit lean:", set(fat_ids.crunchbase_url) == set(lean.crunchbase_url))
pd.read_csv("data/raw/companies_fat.csv", nrows=5, dtype=str).iloc[:, :6]

companies_fat: 25,421 rows x 116 cols
nur in fat (nicht in lean): ['cb_load_date', 'cbcsv_accelerator_application_deadline', 'cbcsv_accelerator_duration_in_weeks', 'cbcsv_accelerator_program_type', 'cbcsv_acquired_by', 'cbcsv_acquired_by_url', 'cbcsv_acquisition_status', 'cbcsv_acquisition_terms', 'cbcsv_acquisition_type', 'cbcsv_actively_hiring', 'cbcsv_announced_date', 'cbcsv_announced_date_precision'] ...
Firmen identisch mit lean: True


,crunchbase_url,cb_source_file,cb_company_name,cb_description,cb_full_description,cb_website
0,https://www.crunchbase.com/organization/0-mission,DK.csv,0-Mission,0-Mission is a B2B marketplace that connects d...,0-Mission is a B2B marketplace that connects d...,https://www.the0mission.dk
1,https://www.crunchbase.com/organization/011h,ES2.csv,011h,011h is a construction company that specialize...,011H specializes in sustainable building devel...,https://www.011h.com/
2,https://www.crunchbase.com/organization/01green,IT1.csv,01Green,01Green is an innovative business focusing on ...,01Green is an innovative business focusing on ...,NaN
3,https://www.crunchbase.com/organization/0credit,BE.csv,byebyerent,Byebyerent offers a shared home equity financi...,At Byebyerent we believe acquiring a home shou...,https://www.byebyerent.com
4,https://www.crunchbase.com/organization/0vix,DE4.csv,0VIX Protocol,0VIX is a core Polygon Money Market Protocol t...,0VIX is a core Polygon Money Market Protocol t...,https://www.0vix.com


In [4]:
# panel_fat: 31 GB, NEVER load fully. A streaming reader (pyarrow) shows the first batch.
from src import data_loading
reader = data_loading.open_panel_fat()
pf = reader.read_next_batch().to_pandas()
print(f"erste Batch: {len(pf):,} rows x {pf.shape[1]} cols")
print("Spalten:", list(pf.columns))
pf.head(4)

erste Batch: 2,910 rows x 128 cols
Spalten: ['crunchbase_url', 'month', 'funding_event', 'funding_n_rounds', 'funding_total_usd', 'funding_types', 'funding_stages', 'funding_n_investors', 'ma_event', 'ma_type', 'ma_acquirer_name', 'ma_price_usd', 'ma_sentiment', 'cb_source_file', 'cb_company_name', 'cb_description', 'cb_full_description', 'cb_website', 'cb_founded_year', 'cb_categories', 'cb_operating_status', 'cb_load_date', 'jina_homepage_content', 'jina_about_content', 'jina_about_url', 'jina_scrape_status', 'jina_scraped_at', 'openai_description', 'openai_described_at', 'perplexity_description', 'perplexity_described_at', 'run_id', 'cbm_type', 'cbm_reasoning', 'cbm_confidence', 'classified_at', 'cbcsv_organization_name', 'cbcsv_company_type', 'cbcsv_industries', 'cbcsv_description', 'cbcsv_headquarters_location', 'cbcsv_headquarters_regions', 'cbcsv_founded_date', 'cbcsv_founded_date_precision', 'cbcsv_patents_granted', 'cbcsv_estimated_revenue_range', 'cbcsv_diversity_spotlight', 

,crunchbase_url,month,funding_event,funding_n_rounds,funding_total_usd,funding_types,funding_stages,funding_n_investors,ma_event,ma_type,ma_acquirer_name,ma_price_usd,ma_sentiment,cb_source_file,cb_company_name,cb_description,cb_full_description,cb_website,cb_founded_year,cb_categories,...,cbcsv_full_description,cbcsv_actively_hiring,cbcsv_number_of_sub_orgs,cbcsv_investor_type,cbcsv_investment_stage,cbcsv_number_of_portfolio_organizations,cbcsv_number_of_investments,cbcsv_number_of_lead_investments,cbcsv_number_of_diversity_investments,cbcsv_number_of_exits,cbcsv_number_of_exits_ipo,cbcsv_accelerator_program_type,cbcsv_accelerator_application_deadline,cbcsv_accelerator_duration_in_weeks,cbcsv_school_type,cbcsv_school_program,cbcsv_number_of_enrollments,cbcsv_school_method,cbcsv_number_of_founders_alumni,cbcsv_number_of_alumni
0,https://www.crunchbase.com/organization/0-mission,2022-01-01,0,0,0,NA,NA,NaN,0,None,None,None,None,DK.csv,0-Mission,0-Mission is a B2B marketplace that connects d...,0-Mission is a B2B marketplace that connects d...,https://www.the0mission.dk,2022,"Energy, Marketplace, Renewable Energy, Small a...",...,0-Mission is a B2B marketplace that connects d...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,https://www.crunchbase.com/organization/0-mission,2022-02-01,0,0,0,NA,NA,NaN,0,None,None,None,None,DK.csv,0-Mission,0-Mission is a B2B marketplace that connects d...,0-Mission is a B2B marketplace that connects d...,https://www.the0mission.dk,2022,"Energy, Marketplace, Renewable Energy, Small a...",...,0-Mission is a B2B marketplace that connects d...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,https://www.crunchbase.com/organization/0-mission,2022-03-01,0,0,0,NA,NA,NaN,0,None,None,None,None,DK.csv,0-Mission,0-Mission is a B2B marketplace that connects d...,0-Mission is a B2B marketplace that connects d...,https://www.the0mission.dk,2022,"Energy, Marketplace, Renewable Energy, Small a...",...,0-Mission is a B2B marketplace that connects d...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,https://www.crunchbase.com/organization/0-mission,2022-04-01,0,0,0,NA,NA,NaN,0,None,None,None,None,DK.csv,0-Mission,0-Mission is a B2B marketplace that connects d...,0-Mission is a B2B marketplace that connects d...,https://www.the0mission.dk,2022,"Energy, Marketplace, Renewable Energy, Small a...",...,0-Mission is a B2B marketplace that connects d...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [5]:
# example firm from the first batch: the raw monthly timeline as it sits in the panel
ex_id = pf.loc[pf["funding_types"].notna(), "crunchbase_url"].iloc[0]
ex_raw = pf[pf["crunchbase_url"] == ex_id]
print("Beispiel-Firma:", ex_id, f"- {len(ex_raw)} Monatszeilen in Batch 1")
show = [c for c in ("crunchbase_url", "month", "n_funding_rounds", "fund_usd", "funding_types") if c in ex_raw.columns]
ex_raw[show].head(10)

Beispiel-Firma: https://www.crunchbase.com/organization/0-mission - 51 Monatszeilen in Batch 1


,crunchbase_url,month,funding_types
0,https://www.crunchbase.com/organization/0-mission,2022-01-01,NA
1,https://www.crunchbase.com/organization/0-mission,2022-02-01,NA
2,https://www.crunchbase.com/organization/0-mission,2022-03-01,NA
3,https://www.crunchbase.com/organization/0-mission,2022-04-01,NA
4,https://www.crunchbase.com/organization/0-mission,2022-05-01,Pre-Seed
5,https://www.crunchbase.com/organization/0-mission,2022-06-01,NA
6,https://www.crunchbase.com/organization/0-mission,2022-07-01,NA
7,https://www.crunchbase.com/organization/0-mission,2022-08-01,NA
8,https://www.crunchbase.com/organization/0-mission,2022-09-01,NA
9,https://www.crunchbase.com/organization/0-mission,2022-10-01,NA


**Why we continue with an event extract:** the 31 GB file is condensed once
into `data/processed/panel_events.parquet` (only months with event content;
2.06 M rows) via `outcomes.extract_panel_events()`, re-run only for a new raw
export. All window outcomes are computed on this extract; completeness against
`panel_lean` is checked by the sanity gate (cross-source check).

## 1.B Cleaning Crunchbase into the firm base (one row per firm)

`src/cohort.py` + `src/sectors.py`. Ground rules enforced here:
- **Closed column list** (`src/schema.py`): no column without declaration and
  justification. End-state snapshots (investors, employees, status, ...) are
  BANNED as covariates or outcomes (leakage), text columns as well.
- **`land` = prefix of the source file, NOT the HQ field.** The exports came
  per country and the HQ column is patchy and inconsistent.
- **Circularity** parsed from the delivered classifier field (conservative
  default, so measured gaps are lower bounds); `parse_error -> 0` exactly
  once, documented.
- **Founding date + precision** (day/month/year). Year-precise firms start in
  January for now (month index `year*12 + month - 1`), enrichment follows in 1.D.

In [6]:
from src import cohort, sectors
base = cohort.build_firm_base()
overview(base, "firm base (nach cohort.build_firm_base)")
print()
print("Land (Quelldatei-Prefix):"); print(base["land"].value_counts().to_string())

firm base (nach cohort.build_firm_base): 25,421 rows x 16 cols
  hoechste Missing-Anteile:
    cbm_confidence                          82.1%
    n_founders                              25.1%
    industries                               2.1%

Land (Quelldatei-Prefix):
land
UK    7669
FR    3152
DE    2698
ES    1949
CH    1614
NL    1331
IT    1254
SE    1186
PL     657
DK     632
FI     551
EE     527
BE     493
NO     472
PT     343
AT     325
LT     243
LV     140
LU     135
IE      50


In [7]:
print("Gruendungskohorten:"); print(base["t_year"].value_counts().sort_index().to_string())
print()
print("Datums-Praezision:"); print(base["founded_precision"].value_counts().to_string())
print()
print("Zirkularitaet:"); print(base["circ"].value_counts().to_string())
print("Subtypen:"); print(base["circ_subtype"].value_counts().to_string())

Gruendungskohorten:
t_year
2016    3317
2017    3392
2018    3367
2019    3418
2020    2988
2021    3339
2022    2328
2023    1789
2024    1124
2025     352
2026       7

Datums-Praezision:
founded_precision
year     14061
day       8542
month     2818

Zirkularitaet:
circ
0    20873
1     4548
Subtypen:
circ_subtype
None        20873
Design       1231
Platform     1096
Nature        876
Service       823
Waste         522


In [8]:
# sectors: the industries tag list becomes n_industries plus top-20 dummies (tag list pinned in schema)
base = sectors.attach_sector_controls(base)
ind_cols = [c for c in base.columns if c.startswith("ind_")]
print(f"+ {len(ind_cols)} Sektor-Dummies; n_industries:")
print(base["n_industries"].describe().round(2).to_string())
print()
print("Top-10 Tag-Anteile:")
print(base[ind_cols].mean().sort_values(ascending=False).head(10).round(3).to_string())

+ 20 Sektor-Dummies; n_industries:
count    24891.0
mean        3.62
std         1.84
min          1.0
25%          2.0
50%          3.0
75%          4.0
max         15.0

Top-10 Tag-Anteile:
ind_software                  0.240
ind_ai                        0.143
ind_information_technology    0.137
ind_health_care               0.096
ind_saas                      0.089
ind_financial_services        0.073
ind_fintech                   0.071
ind_ecommerce                 0.059
ind_biotechnology             0.059
ind_manufacturing             0.054


## 1.C Cleaning the panel into 36-month outcomes

`src/outcomes.py`: window = founding month + 35 months. Grant rounds are
separable via `funding_types` (100 % of event months labelled), so
`fund36_private` exists as a grant-free outcome. **Exclusion with reason:**
events BEFORE the founding month (a CB artifact, 267 firms affected) fall
outside the window by definition. Mixed months (private + grant) count as
private in the amount sums.

In [9]:
from src import outcomes
outcomes.extract_panel_events(force=False)          # no-op while the extract exists
panel = outcomes.load_panel_events()
print(f"panel_events: {len(panel):,} Zeilen (Firma x Monat mit Ereignis-Inhalt)")
panel.head(4)

panel_events: 2,063,279 Zeilen (Firma x Monat mit Ereignis-Inhalt)


,crunchbase_url,month,funding_event,funding_total_usd,funding_types,ma_event,midx,private_month,grant_month
0,https://www.crunchbase.com/organization/0-mission,2022-01-01,0,0.0,None,0,24264,0,0
1,https://www.crunchbase.com/organization/0-mission,2022-02-01,0,0.0,None,0,24265,0,0
2,https://www.crunchbase.com/organization/0-mission,2022-03-01,0,0.0,None,0,24266,0,0
3,https://www.crunchbase.com/organization/0-mission,2022-04-01,0,0.0,None,0,24267,0,0


In [10]:
# window logic demonstrated on the example firm (panel_events: midx = month index)
exb = base[base["firm_id"] == ex_id]
if len(exb):
    f0 = int(exb["founding_month_idx"].iloc[0])
    ev = panel[panel["crunchbase_url"] == ex_id].sort_values("midx")
    ev = ev.assign(offset_m=ev["midx"] - f0, im_fenster=lambda d: d["offset_m"].between(0, outcomes.MAIN_WINDOW - 1))
    print(f"{ex_id}: Gruendungsindex {f0} -> Fenster [{f0}, {f0 + outcomes.MAIN_WINDOW - 1}]")
    display(ev[["month", "midx", "offset_m", "im_fenster", "funding_event",
                "funding_total_usd", "funding_types", "private_month", "grant_month"]].head(8))
else:
    print(ex_id, "liegt ausserhalb der spaeteren Kohorten - Demo uebersprungen")

https://www.crunchbase.com/organization/0-mission: Gruendungsindex 24264 -> Fenster [24264, 24299]


,month,midx,offset_m,im_fenster,funding_event,funding_total_usd,funding_types,private_month,grant_month
0,2022-01-01,24264,0,True,0,0.0,None,0,0
1,2022-02-01,24265,1,True,0,0.0,None,0,0
2,2022-03-01,24266,2,True,0,0.0,None,0,0
3,2022-04-01,24267,3,True,0,0.0,None,0,0
4,2022-05-01,24268,4,True,1,130000.0,Pre-Seed,1,0
5,2022-06-01,24269,5,True,0,0.0,None,0,0
6,2022-07-01,24270,6,True,0,0.0,None,0,0
7,2022-08-01,24271,7,True,0,0.0,None,0,0


In [11]:
agg = outcomes.aggregate_panel_windows(base, panel)
base = outcomes.attach_outcomes(base, agg)          # 1:1-Merge auf firm_id (validiert)
rate_cols = ["fund36", "fund36_private", "grant36_cb", "success36"]
print("Outcome-Raten (alle 25.421 Firmen, vor Row-Policy):")
print(base[rate_cols].astype("float64").mean().round(4).to_string())

      note: 343 in-window months carry BOTH a private round and a grant, counted as private in amount sums (rule in src/outcomes.py)
Outcome-Raten (alle 25.421 Firmen, vor Row-Policy):
fund36            0.7516
fund36_private    0.7139
grant36_cb        0.0997
success36         0.1422


## 1.D Loading, cleaning and matching registry/crawl data (founding months v2)

For 55 % of the firms the CB founding date is year-precise only. Enrichment
from official registers (UK CH, FR Sirene, BE KBO, DK CVR/OpenCorporates, FI,
NO, EE, LV, IE, LU) plus verified web months. Kennedy et al. (2024) likewise
fill missing founding information from "Pitchbook, Crunchbase and
OpenCorporates"; founding = register entry following Guzman & Kacperczyk (2019).

Match discipline (as in the grant match): exact normalized name plus a second
independent signal (geo, year or uniqueness) gives an auto-confirm; anything
uncertain stays out or is reviewed pair by pair (append-only verdict store).
Month adoption: register year == founding year, plus a **Q1 lag rule**
(register entry in Jan-Mar of the following year = registration lag).

In [12]:
from src import enrich
before_imp = float((base["founded_precision"] == "year").mean())
base = enrich.attach_founding_v2(base)
base = enrich.attach_v2_outcomes(base, panel)
print()
print("v2-Quellenmix:"); print(base["founding_v2_source"].value_counts().to_string())
print()
print(f"jahresgenau (vorher):        {before_imp:6.1%}")
print(f"Januar-imputiert unter v2:   {float(base['founding_month_imputed_v2'].mean()):6.1%}")

      founding v2 sources: {'cb_day': 8542, 'january': 7268, 'cb_month': 2818, 'registry_uk': 1887, 'registry_fr': 1210, 'registry_de': 757, 'web': 632, 'registry_es': 450, 'registry_se': 350, 'registry_nl': 345, 'registry_it': 296, 'registry_ch': 233, 'registry_dk': 183, 'registry_fi': 120, 'registry_no': 100, 'registry_be': 94, 'registry_ee': 54, 'registry_pt': 35, 'registry_at': 27, 'registry_ie': 11, 'registry_lu': 7, 'registry_lv': 2}
      imputed share: 55.3% (v1) -> 28.6% (v2); months gained: 6,793



v2-Quellenmix:
founding_v2_source
cb_day         8542
january        7268
cb_month       2818
registry_uk    1887
registry_fr    1210
registry_de     757
web             632
registry_es     450
registry_se     350
registry_nl     345
registry_it     296
registry_ch     233
registry_dk     183
registry_fi     120
registry_no     100
registry_be      94
registry_ee      54
registry_pt      35
registry_at      27
registry_ie      11
registry_lu       7
registry_lv       2

jahresgenau (vorher):         55.3%
Januar-imputiert unter v2:    28.6%


In [13]:
# the Q1 lag rule made visible: adopted months in Q1 of the following year
v2y = base["founding_month_idx_v2"] // 12
lag = (v2y == base["t_year"] + 1)
print(f"Monate via Q1-Lag-Regel (Jan-Maer des Folgejahres): {int(lag.sum())}")
print(base.loc[lag, "founding_v2_source"].value_counts().to_string())

Monate via Q1-Lag-Regel (Jan-Maer des Folgejahres): 283
founding_v2_source
registry_uk    77
registry_fr    54
registry_de    31
registry_es    30
registry_nl    21
registry_se    21
registry_ch    13
registry_ee     8
registry_it     5
registry_dk     5
registry_no     5
registry_be     4
registry_fi     3
registry_ie     3
registry_at     1
registry_pt     1
registry_lu     1


## 1.E Loading, cleaning and matching the grant data (CORDIS)

Two upstream steps (scripts, only consumed here): `match_cordis.py` (raw
candidates against all 25,421 firms) and `validate_grant_match.py finalize`
(merge of the pair-level verdicts). **Exclusions with reason:** rejected pairs
(wrong company, e.g. name lookalikes) and uncertain pairs NEVER enter treated
status, only `grant_match_final.csv` counts. First grant by `ecSignatureDate`
(100 % filled for SME/EIC); `-BF` schemes (EIC equity) stay separable via
`first_grant_scheme`.

In [14]:
verd = pd.read_csv("outputs/tables/grant_match_final.csv")
print("Pair-Verdicts (firm x organisation):")
print(verd["final_status"].value_counts().to_string())
from src import grants
base = grants.attach_grants(base)
tr = base[base["has_grant"] == 1]
print(f"\nTreated-Firmen (verified): {len(tr):,}")
print(tr["first_grant_scheme_group"].value_counts().to_string())

Pair-Verdicts (firm x organisation):
final_status
verified_auto    1214
verified_llm      524
rejected          217
uncertain          23


      grant block: 1,723 treated firms | 581 circular | 19 uncertain-only (untreated)

Treated-Firmen (verified): 1,723
first_grant_scheme_group
consortium-R&D    874
SME-Instrument    536
EIC               313


## 1.F Survival snapshot, merge and row policy

`src/survival.py`: the status columns (closed/exit/site_dead) are the ONLY
permitted end-state exception, outcome-only and never a covariate.

Then the **only two row drops** of the build, both with a reason:
1. **Cohorts 2016-2022**: complete 36m windows up to the panel end (2026-03)
   and uniform classifier coverage.
2. **Present in panel_fat**: without panel rows there are no dated outcomes;
   the 110 in-cohort firms without panel rows are dropped and documented.

In [15]:
from src import survival
base = survival.attach_survival(base)
coh = base["t_year"].between(cohort.COHORT_MIN, cohort.COHORT_MAX)
print(f"vor Row-Policy:            {len(base):,} Firmen")
print(f"ausserhalb 2016-2022:      {int((~coh).sum()):,}  (Drop 1: unvollstaendige Fenster/Kohorten-Scope)")
absent = coh & ~base["in_panel"]
print(f"in-cohort, ohne Panel:     {int(absent.sum()):,}  (Drop 2: keine datierten Outcomes moeglich)")
df = base[coh & base["in_panel"]].copy()
print(f"= Analyse-Dataframe:       {len(df):,} Firmen")
dropped = pd.read_csv("outputs/tables/dropped_panel_absent_firms.csv")
print(f"\nDrop-2-Liste (dokumentiert, {len(dropped)} Firmen, davon funded im sealed Artifact: {int(dropped['fund36'].fillna(0).sum())}):")
dropped.head(5)

vor Row-Policy:            25,421 Firmen
ausserhalb 2016-2022:      3,272  (Drop 1: unvollstaendige Fenster/Kohorten-Scope)
in-cohort, ohne Panel:     110  (Drop 2: keine datierten Outcomes moeglich)
= Analyse-Dataframe:       22,039 Firmen

Drop-2-Liste (dokumentiert, 110 Firmen, davon funded im sealed Artifact: 79):


,firm_id,cb_company_name,land,t_year,circ,fund36,success36
0,https://www.crunchbase.com/organization/apohem,Apohem,SE,2017,0,1,0
1,https://www.crunchbase.com/organization/apollo...,Apollo Therapeutics,UK,2016,0,0,0
2,https://www.crunchbase.com/organization/apolo-p2p,APOLO P2P,EE,2022,0,1,0
3,https://www.crunchbase.com/organization/aponomy,Aponomy,SE,2016,0,1,1
4,https://www.crunchbase.com/organization/apoqlar,apoQlar,DE,2017,0,0,0


## 1.G Validation and gates

Three layers: (1) schema, exactly the 76 declared columns in fixed order, no
undeclared column can enter; (2) funnel, reproduced and checked against the
pipeline artifact; (3) an **equality assert against
`data/processed/firms_analysis.parquet`**: this notebook produces exactly the
dataset of the pipeline, nothing is lost. After that the pinned sanity checks
(anchor rates, cross-source sample against panel_lean) as in the script gate.

In [16]:
from src import schema
from build_analysis import cleaning_funnel
df = df.copy()
for c in ("n_fund_events_36", "fund36", "fund36_private", "grant36_cb", "success36"):
    df[c] = df[c].astype("int16" if c == "n_fund_events_36" else "int8")
funnel = cleaning_funnel(base, df)
df = df.drop(columns=["in_panel"])[schema.expected_columns()]
schema.validate(df)
print("Schema: 76 Spalten, Reihenfolge exakt - ok")
funnel_pipeline = pd.read_csv("outputs/tables/cleaning_funnel.csv")
assert funnel.reset_index(drop=True).equals(funnel_pipeline), "Funnel weicht ab!"
print("Funnel identisch mit outputs/tables/cleaning_funnel.csv - ok")
funnel

Schema: 76 Spalten, Reihenfolge exakt - ok
Funnel identisch mit outputs/tables/cleaning_funnel.csv - ok


,stage,total,circ,non_circ
0,all firms (raw),25421,4548,20873
1,founding cohorts 2016-2022,22149,4017,18132
2,+ present in panel_fat (= analysis dataframe),22039,3997,18042
3,of which verified EU grant (has_grant),1661,559,1102
4,of which SME-Instrument/EIC first grant,834,263,571


In [17]:
# the core proof: cell for cell identical to the pipeline parquet
pipe = pd.read_parquet("data/processed/firms_analysis.parquet")
a = df.sort_values("firm_id").reset_index(drop=True)
b = pipe.sort_values("firm_id").reset_index(drop=True)
pd.testing.assert_frame_equal(a, b, check_dtype=False)
print(f"IDENTISCH: Notebook-Build == data/processed/firms_analysis.parquet "
      f"({len(a):,} rows x {a.shape[1]} cols)")

IDENTISCH: Notebook-Build == data/processed/firms_analysis.parquet (22,039 rows x 76 cols)


/var/folders/g8/y67mzh1j4y11s_6rlfzwpl4c0000gn/T/ipykernel_21303/1008455151.py:5: FutureWarning: Mismatched null-like values nan and None found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  pd.testing.assert_frame_equal(a, b, check_dtype=False)


In [18]:
import sanity_checks
sanity_checks.run_all(df, panel=panel)

Hard checks:
  [ok ] schema: exact closed column list
  [ok ] N == 22,039 (got 22,039)
  [ok ] firm_id unique
  [ok ] t_year ∈ [2016, 2022]
  [ok ] land counts match pinned values exactly
  [ok ] circ_subtype counts match (got {'Design': 1067, 'Platform': 970, 'Nature': 783, 'Service': 727, 'Waste': 450})
  [ok ] non-circular ⇒ circ_subtype == 'None'
  [ok ] circ == 3,997
  [ok ] parse_error == 22 (coded non-circular)
  [ok ] founded-date precision counts match
  [ok ] n_industries consistent with raw industries string
  [ok ] industry dummies are 0/1 and 0 without tags
  [ok ] fund36 == 1[n_fund_events_36 ≥ 1]
  [ok ] fund36_private ⊆ fund36
  [ok ] grant36_cb ⊆ fund36
  [ok ] log outcomes == log1p(sums)
  [ok ] private amounts ≤ total amounts
  [ok ] fund36_private == 1 ⇒ first private round inside the window
  [ok ] success36 == OR of its three components
  [ok ] verified grant firms == 1,661
  [ok ] circular grant firms == 559
  [ok ] uncertain-only grant firms == 16 (untreated)
  

  [ok ] pre-founding events == 267 (CB-artifact pin, analysis firms; got 267)


      note: 281 in-window months carry BOTH a private round and a grant, counted as private in amount sums (rule in src/outcomes.py)
  [ok ] all outcome columns reproduce from panel_events.parquet


  [ok ] v2 outcome columns reproduce under founding_month_idx_v2



Anchor check (cohorts 2016-21; pins re-based on analysis rows):
  rate                 full(25,421)  pin(22,039)  jan-replication  precision-aware
  fund36_circ                 64.5%       64.43%           64.43%           65.88%
  fund36_noncirc              70.8%       70.79%           70.79%           72.25%
  success36_circ              13.6%       13.62%           13.62%           15.58%
  success36_noncirc           12.9%       12.93%           12.93%           14.50%
  [ok ] all anchor rates equal their re-based pins (round-4 exact)


  [ok ] panel_lean rows == 2,073,807 (raw drift guard)
  [ok ] cross-source check: 2,000-firm seeded sample identical in panel_lean (seed 20260711)

Funding rates (analysis rows, cohorts 2016-2022):
  fund36          circ  68.7%   non-circ  74.0%
  fund36_private  circ  64.2%   non-circ  70.6%
  grant36_cb      circ  14.1%   non-circ   8.9%
  success36       circ  16.6%   non-circ  14.3%
  has_grant       circ  14.0%   non-circ   6.1%

First-grant scheme groups (verified match):
                          firms  circ  mono  median_eur
first_grant_scheme_group                               
EIC                         301   116  0.79   2493012.0
SME-Instrument              533   147  0.96     50000.0
consortium-R&D              827   296  0.09    280014.0

Time to first PRIVATE round (months from founding, uncensored, n=20,197): P50=18  P75=35  P90=54

SANITY GATE GREEN: all checks passed.


### Matched grant samples (building blocks for RQ2, script gates as before)
For each treated firm up to 4 never-treated twins from the same cell
(cohort × circ × country × pre-grant history incl. 12m momentum) with a
pseudo grant date, following Croce et al. (2013); balance tables below.

In [19]:
run("conda run -n ml python scripts/build_grants_sample.py")

   sme_eic          pre_n_private        0.9235        0.9391   -0.0139
   sme_eic      pre12_any_private        0.3880        0.3889   -0.0020
   sme_eic             n_founders        2.2756        2.1297    0.1301
   sme_eic           n_industries        3.5151        3.7120   -0.1043
   sme_eic founding_month_imputed        0.6762        0.5804    0.1991
   sme_eic            grant_age_m       31.7760       31.4861    0.0161
consortium          pre_n_private        1.1424        1.1673   -0.0210
consortium      pre12_any_private        0.3585        0.3585    0.0000
consortium             n_founders        2.3151        2.0967    0.1889
consortium           n_industries        3.6511        3.5917    0.0332
consortium founding_month_imputed        0.5918        0.5794    0.0251
consortium            grant_age_m       39.7804       39.4086    0.0183


In [20]:
run("conda run -n ml python scripts/build_grants_sample.py --fidx founding_month_idx_v2 --out grants_matched_v2.csv")

   sme_eic          pre_n_private        0.9248        0.9368   -0.0107
   sme_eic      pre12_any_private        0.3885        0.3907   -0.0045
   sme_eic             n_founders        2.2777        2.1573    0.1088
   sme_eic           n_industries        3.5144        3.7578   -0.1278
   sme_eic founding_month_imputed        0.1655        0.2691   -0.2532
   sme_eic            grant_age_m       29.2955       28.9996    0.0167
consortium          pre_n_private        1.1503        1.1955   -0.0379
consortium      pre12_any_private        0.3610        0.3635   -0.0052
consortium             n_founders        2.3143        2.1083    0.1770
consortium           n_industries        3.6539        3.6888   -0.0190
consortium founding_month_imputed        0.1468        0.2682   -0.3025
consortium            grant_age_m       37.7202       37.4209    0.0148


## 1.H Variable catalogue: what ends up inside?

All 76 columns with definition, source and cleaning note (generated from
`src/schema.py`, the same source as the codebook in the data package), then
descriptives over all numeric columns and the missingness of the final dataset.

In [21]:
cat = pd.DataFrame([{"spalte": c.name, "gruppe": c.group, "dtype": c.dtype,
                     "definition": c.definition, "quelle": c.source,
                     "cleaning": c.cleaning} for c in schema.SCHEMA])
print(cat["gruppe"].value_counts().to_string())
pd.set_option("display.max_rows", 90); pd.set_option("display.max_colwidth", 72)
cat

gruppe
Team & sector controls                                         23
Outcomes 36m                                                   13
EU grants (verified CORDIS match)                               9
Founding v2 (registry/web enrichment, parallel columns)         6
Outcomes 36m under the v2 founding month (parallel columns)     6
Founding                                                        5
Circularity                                                     5
Survival & exit (status at data cutoff, outcome-only)           5
Identity                                                        4


,spalte,gruppe,dtype,definition,quelle,cleaning
0,firm_id,Identity,string,Unique firm key,lean.crunchbase_url,
1,cb_company_name,Identity,string,Company name (QA convenience),lean.cb_company_name,
2,cb_source_file,Identity,string,Scrape source file (audit trail for land),lean.cb_source_file,
3,land,Identity,string,"Sampling-frame country, 20 values",lean.cb_source_file,"leading letter prefix, e.g. 'UK_2018_1' → 'UK'; asserted ⊆ 20-land set"
4,founded_date,Founding,date,Founding date,fat.cbcsv_founded_date,100% filled; parse errors=raise
5,founded_precision,Founding,string,day / month / year,fat.cbcsv_founded_date_precision,
6,t_year,Founding,int16,"Founding year (analysis cohorts 2016–2022, rows outside dropped, see...",year(founded_date),asserted == lean.cb_founded_year for all rows
7,founding_month_idx,Founding,int32,Founding month as integer index year*12+(month-1),founded_date,year-precision firms imputed to January of t_year
8,founding_month_imputed,Founding,int8 0/1,1 = founding month imputed (precision == 'year'),derived,
9,cbm_type,Circularity,string,Raw business-model type incl. 'parse_error',lean.cbm_type,


In [22]:
num = df.select_dtypes("number")
num.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
t_year,22039.0,2018.845093,1.942535,2016.0,2017.0,2019.0,2021.0,2022.0
founding_month_idx,22039.0,24228.30691,23.556951,24192.0,24205.0,24228.0,24252.0,24275.0
founding_month_imputed,22039.0,0.554335,0.49705,0.0,0.0,1.0,1.0,1.0
circ,22039.0,0.18136,0.385325,0.0,0.0,0.0,0.0,1.0
parse_error,22039.0,0.000998,0.03158,0.0,0.0,0.0,0.0,1.0
cbm_confidence,3997.0,0.802402,0.07642,0.6,0.7,0.8,0.9,0.9
n_founders,16942.0,2.083992,1.074781,1.0,1.0,2.0,3.0,18.0
n_industries,21698.0,3.647479,1.856175,1.0,2.0,3.0,4.0,15.0
ind_software,22039.0,0.23717,0.425357,0.0,0.0,0.0,0.0,1.0
ind_ai,22039.0,0.118699,0.323441,0.0,0.0,0.0,0.0,1.0


In [23]:
miss = df.isna().mean().sort_values(ascending=False)
print("Missingness im finalen Datensatz (nur Spalten mit Luecken):")
print(miss[miss > 0].round(4).to_string())
print("\nAlle uebrigen Spalten sind vollstaendig.")

Missingness im finalen Datensatz (nur Spalten mit Luecken):
first_ma_offset_m            0.9664
registry_dissolution_date    0.9656
first_grant_scheme           0.9246
first_grant_scheme_group     0.9246
ec_total_eur                 0.9246
n_grant_participations       0.9246
grant_match_tier             0.9246
first_grant_date             0.9246
cbm_confidence               0.8186
registry_status              0.5052
registry_id                  0.4203
n_founders                   0.2313
first_private_offset_m_v2    0.0871
first_private_offset_m       0.0836
industries                   0.0155
n_industries                 0.0155
site_dead_2026_03            0.0049

Alle uebrigen Spalten sind vollstaendig.


**Result:** `data/processed/firms_analysis.parquet` (+ CSV mirror),
`grants_matched.csv`, `grants_matched_v2.csv`, identical to the one-command
pipeline build (`run_pipeline.sh`). Models: the numbered analysis sequence
(`2_descriptive_statistics.R` to `8_frame_robustness.R`).